# PRÉ-PROCESSAMENTO COM O DATASET ZOO

Dataset obtido via repositório UCI Machine Learning (`ucimlrepo`).
Seguindo as etapas de pré-processamento de dados: exploração técnica, verificação de dados faltantes e tratamento de outliers com método IQR e capping.

In [ ]:
# 0. Importando as Bibliotecas Necessárias
import ssl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

# Ajuste de contexto SSL para download do dataset sem bloqueios de certificado
ssl._create_default_https_context = ssl._create_unverified_context

# Configurando o estilo dos gráficos para melhor visualização
sns.set_style('whitegrid')

## Passo 1: Investigando a Cena do Crime (os Dados)
Carregamos o dataset Zoo (ID: 111) e inspecionamos as primeiras linhas, resumo técnico e registros faltantes.

In [ ]:
# Carregando o dataset Zoo (ID: 111)
zoo = fetch_ucirepo(id=111)
df_zoo = zoo.data.original.copy()

print("As 5 primeiras amostras do dataset:")
df_zoo.head()

In [ ]:
# Resumo técnico do dataset
print("Relatório técnico do dataset:")
df_zoo.info()

In [ ]:
# Contagem de registros corrompidos (dados faltantes)
print("Contagem de registros faltantes por coluna:")
df_zoo.isnull().sum()

## Passo 2: Consertando as Medidas Faltantes
Estratégia de cálculo de mediana para colunas numéricas e preenchimento de eventuais dados faltantes via `.fillna()`.

In [ ]:
colunas_numericas = df_zoo.select_dtypes(include=[np.number]).columns
medianas = df_zoo[colunas_numericas].median()

df_zoo.fillna(medianas, inplace=True)

print("Verificação após o primeiro conserto (dados faltantes):")
df_zoo.isnull().sum()

## Passo 3: Encontrando Outliers
No dataset Zoo, os atributos de características físicas são predominantemente binários (0 ou 1: pelo, pena, ovos, etc.).
A coluna `legs` (número de patas/pernas) é uma medida contínua/discreta variando de 0 a 8 pernas, ideal para análise de outliers.

In [ ]:
# Boxplot do atributo legs ANTES do tratamento
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_zoo['legs'])
plt.title('Boxplot de legs (Número de Pernas) - ANTES do Tratamento')
plt.show()

In [ ]:
# Análise de outliers via método IQR
Q1 = df_zoo['legs'].quantile(0.25)
Q3 = df_zoo['legs'].quantile(0.75)
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR
limite_inferior = Q1 - 1.5 * IQR

outliers = df_zoo[(df_zoo['legs'] > limite_superior) | (df_zoo['legs'] < limite_inferior)]
outliers_count = outliers.shape[0]

print(f"Q1 (25%): {Q1}")
print(f"Q3 (75%): {Q3}")
print(f"IQR: {IQR}")
print(f"Limite superior para outliers: {limite_superior:.2f}")
print(f"Total de outliers identificados: {outliers_count}")

if outliers_count > 0:
    print("\nAnimais considerados outliers estatísticos pelo método IQR:")
    display(outliers[['animal_name', 'legs']])

In [ ]:
# Tratamento com Capping (substituindo valores acima do limite superior)
df_zoo_capped = df_zoo.copy()
df_zoo_capped['legs'] = np.where(
    df_zoo_capped['legs'] > limite_superior,
    limite_superior,
    df_zoo_capped['legs']
)
print("Outliers em 'legs' tratados com sucesso via técnica de Capping.")

## Passo 4: Inspeção Final
Verificamos a nova distribuição com o boxplot após o capping e as estatísticas descritivas finais.

In [ ]:
# Boxplot de legs APÓS o tratamento
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_zoo_capped['legs'])
plt.title('Boxplot de legs (Número de Pernas) - APÓS o Tratamento')
plt.show()

In [ ]:
# Estatísticas descritivas do dataset limpo
print("Estatísticas descritivas do dataset Zoo limpo:")
df_zoo_capped.describe()